# Figures S9, S10 & S11: ACC-HPC Reinstatement (RSA) vs Subject-Level Covariates, Memory Performance & ACC Reactivation

| Field | Value |
|---|---|
| **Manuscript panels / stats** | Fig S9A (ACC-HPC RSA vs ACC & HPC electrode counts: Spearman r = -0.38, -0.10, ps > 0.05), Fig S9B (ACC-HPC RSA vs RW learning rate & inverse temperature: Spearman r = 0.04, -0.05, ps > 0.05), Fig S10 (ACC-HPC RSA vs recognition d': Spearman r = -0.07, p > 0.05), Fig S11 (ACC-HPC RSA vs ACC RPE reactivation: Spearman r = -0.35, p > 0.05) |
| **Inputs** | `group_results_ACC_to_HPC.pkl`, `ACC_rpe_category_HFA_subject_reinstatement.csv`, `learn_df_RWH_v2.csv`, `full_df_RWH_v2.csv`, `Day1_reref_elec_df` |
| **Outputs** | `results/acc_hpc_rsa_covariates/*`, `figures/FigS9_S10_S11_RSA_covariates/{FigS9_S10_subject_covariates_scatter.pdf,FigS11_acc_reactivation_vs_hit_rsa_scatter.pdf}` |
| **Dependencies** | `membandit`, `mne`, `numpy`, `pandas`, `scipy`, `matplotlib`, `seaborn` |
| **Expected runtime** | < 30 seconds |
| **Upstream steps** | `04_decoding_rsa_followups/Fig4_S8_RSA.ipynb` and `matlab/test_script_crossdecoding.m` |


In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to $MEMORYBANDIT_DATA_DIR)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


## ACC-HPC Hit RSA vs Subject-Level Covariates & ACC RPE Reactivation (Figures S9, S10, S11)

This notebook duplicates the Granger-causality correlation workflow from
`MemoryBandit-ACC_dmPFC_Reinstatement_GC_RPE.ipynb`, but replaces subject-level
ACC reinstatement with subject-level ACC→HPC hit RSA. The RSA target is computed by
reconstructing the significant ACC→HPC remembered-minus-forgotten cluster from
`HFA_RPE_RSA.ipynb`, then averaging each subject's hit/remembered RSA within that exact
cluster mask. The resulting subject-level hit RSA is then correlated with condition-specific
theta-band GC (`gc_net_z`) from **cue_on** epochs using fixed cue windows and a sliding-window
time-resolved GC analysis.


In [ ]:
import json
import os
import pickle
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from mne.stats import permutation_cluster_1samp_test
from scipy.sparse import csr_matrix, eye, kron
from scipy.stats import pearsonr, spearmanr


In [ ]:
from membandit import paths as mpaths

corr_type = 'spearman'
run_secondary_pearson = True
label_points = True

hit_memory_label = 'remembered'
rsa_metric_name = 'HitRSA_cluster_mean'
rsa_target_label = 'ACC-HPC Hit RSA'
rsa_metric_col = f'ACC_HPC_{rsa_metric_name}'

acc_reactivation_metric = 'MeanAccuracy'
acc_reactivation_label = 'ACC Reactivation'
acc_reactivation_col = f'ACC_{acc_reactivation_metric}'
acc_reactivation_analysis_label = 'ACC Reactivation vs ACC-HPC Hit RSA'

rsa_feedback_roi = 'ACC'
rsa_cue_roi = 'HPC'

base_dir = mpaths.get_base_dir()
load_dir = mpaths.get_emu_behavior_dir()

_rsa_candidates = [
    f'{base_dir}/scratch/MemoryBandit/RSA_MVTemplate/group_level/group_results_ACC_to_HPC.pkl',
    f'{base_dir}/scratch/MemoryBandit/RSA/crossregion/group_results_ACC_to_HPC.pkl',
]
rsa_group_results_pkl = next((p for p in _rsa_candidates if os.path.isfile(p)), _rsa_candidates[0])

acc_reinstatement_csv = (
    f'{base_dir}/scratch/MemoryBandit/MVPA/crossdecoding/rpe_category/'
    'ACC_rpe_category_HFA_subject_reinstatement.csv'
)
hpc_theta_reinstatement_csv = (
    f'{base_dir}/scratch/MemoryBandit/MVPA/decoding_cue/hit_miss/'
    'HPC_memory_outcome_hit_miss_theta_subject_reinstatement.csv'
)
hpc_hfa_reinstatement_csv = hpc_theta_reinstatement_csv.replace('theta', 'hfa')

learn_df_csv = f'{load_dir}/learn_df_RWH_v2.csv'
if not os.path.isfile(learn_df_csv):
    learn_df_csv = f'{load_dir}/learn_df_RWH.csv'
combined_df_csv = f'{load_dir}/full_df_RWH_v2.csv'
if not os.path.isfile(combined_df_csv):
    combined_df_csv = f'{load_dir}/full_df_RWH.csv'
combined_df_condition = 'Day1'

cluster_alpha = 0.05
cluster_n_permutations = 5000  # Matches Methods: 5,000 permutations, one-tailed
cluster_seed = 42
cluster_tail = 1

analysis_output_root = Path(mpaths.results_root()) / 'acc_hpc_rsa_covariates'
figure_output_root = Path(mpaths.figure_output_root()) / 'FigS9_S10_S11_RSA_covariates'
analysis_output_root.mkdir(parents=True, exist_ok=True)
figure_output_root.mkdir(parents=True, exist_ok=True)

analysis_prefix = 'ACC_HPC_hits_cluster_mean_covariates'
paths: Dict[str, object] = {
    'subject_metric_csv': analysis_output_root / f'{analysis_prefix}_subject_metric.csv',
    'cluster_metadata_csv': analysis_output_root / f'{analysis_prefix}_cluster_metadata.csv',
    'cluster_metadata_json': analysis_output_root / f'{analysis_prefix}_cluster_metadata.json',
    'acc_reactivation_merge_csv': analysis_output_root / f'{analysis_prefix}_acc_reactivation_vs_hit_rsa_subjects.csv',
    'acc_reactivation_results_csv': analysis_output_root / f'{analysis_prefix}_acc_reactivation_vs_hit_rsa_results.csv',
    'subject_covariate_merge_csv': analysis_output_root / f'{analysis_prefix}_subject_covariates_merged_subjects.csv',
    'subject_covariate_results_csv': analysis_output_root / f'{analysis_prefix}_subject_covariates_results.csv',
    'subject_covariate_summary_txt': analysis_output_root / f'{analysis_prefix}_subject_covariates_summary.txt',
    'hpc_theta_merge_csv': analysis_output_root / f'{analysis_prefix}_hpc_theta_mean_accuracy_merged_subjects.csv',
    'hpc_theta_results_csv': analysis_output_root / f'{analysis_prefix}_hpc_theta_mean_accuracy_results.csv',
    'hpc_hfa_merge_csv': analysis_output_root / f'{analysis_prefix}_hpc_hfa_mean_accuracy_merged_subjects.csv',
    'hpc_hfa_results_csv': analysis_output_root / f'{analysis_prefix}_hpc_hfa_mean_accuracy_results.csv',
    'acc_reactivation_scatter_pdf': figure_output_root / 'FigS11_acc_reactivation_vs_hit_rsa_scatter.pdf',
    'subject_covariate_scatter_pdf': figure_output_root / 'FigS9_S10_subject_covariates_scatter.pdf',
    'hpc_theta_scatter_pdf': figure_output_root / 'hpc_theta_mean_accuracy_vs_hit_rsa_scatter.pdf',
    'hpc_hfa_scatter_pdf': figure_output_root / 'hpc_hfa_mean_accuracy_vs_hit_rsa_scatter.pdf',
}

subject_covariate_predictor_order = [
    'alpha',
    'beta',
    'total_reward',
    'dprime',
    'ACC_NumElectrodes',
    'HPC_NumElectrodes',
]
subject_covariate_predictor_labels = {
    'alpha': 'Alpha',
    'beta': 'Beta',
    'total_reward': 'Total reward',
    'dprime': 'd-prime',
    'ACC_NumElectrodes': 'ACC electrode count',
    'HPC_NumElectrodes': 'HPC electrode count',
}

print('Analysis output root:', analysis_output_root)
print('Figure output root:', figure_output_root)


In [ ]:
def assert_required_columns(tbl: pd.DataFrame, required_cols: List[str], table_name: str) -> None:
    missing = [c for c in required_cols if c not in tbl.columns]
    if missing:
        raise ValueError(f"{table_name} is missing required columns: {missing}")


def deterministic_inner_join(left: pd.DataFrame, right: pd.DataFrame, keys: List[str]) -> pd.DataFrame:
    out = left.merge(right, on=keys, how='inner')
    return out.sort_values(keys).reset_index(drop=True)


def _extract_corr(res) -> Tuple[float, float]:
    if hasattr(res, 'statistic') and hasattr(res, 'pvalue'):
        return float(res.statistic), float(res.pvalue)
    return float(res[0]), float(res[1])


def run_correlation_analysis(
    x: np.ndarray,
    y: np.ndarray,
    corr_type: str = 'spearman',
    run_secondary_pearson: bool = True,
) -> Dict[str, float]:
    mask = np.isfinite(x) & np.isfinite(y)
    x_valid = x[mask]
    y_valid = y[mask]
    n = int(len(x_valid))
    if n < 3:
        raise ValueError(f'Need at least 3 valid observations; got {n}.')

    rho_s, p_s = _extract_corr(spearmanr(x_valid, y_valid, nan_policy='omit'))
    if run_secondary_pearson or corr_type.lower() == 'pearson':
        r_p, p_p = _extract_corr(pearsonr(x_valid, y_valid))
    else:
        r_p, p_p = np.nan, np.nan

    corr_type_norm = corr_type.lower()
    if corr_type_norm == 'spearman':
        primary_r, primary_p = rho_s, p_s
    elif corr_type_norm == 'pearson':
        primary_r, primary_p = r_p, p_p
    else:
        raise ValueError(f"Unsupported corr_type: {corr_type}. Use 'spearman' or 'pearson'.")

    return {
        'n': n,
        'spearman_rho': rho_s,
        'spearman_p': p_s,
        'pearson_r': r_p,
        'pearson_p': p_p,
        'primary_corr_type': corr_type_norm,
        'primary_r': primary_r,
        'primary_p': primary_p,
    }


def _fit_and_plot_line(ax, x: np.ndarray, y: np.ndarray) -> None:
    if len(x) < 2 or len(np.unique(x)) < 2:
        return
    coeff = np.polyfit(x, y, 1)
    xfit = np.linspace(float(np.min(x)), float(np.max(x)), 100)
    yfit = np.polyval(coeff, xfit)
    ax.plot(xfit, yfit, color='#222222', linewidth=1.2)


def plot_roi_condition_grid(
    merged_df: pd.DataFrame,
    results_df: pd.DataFrame,
    target_col: str,
    target_label: str,
    gc_col: str,
    conditions: List[Tuple[str, str]],
    output_pdf: Path,
    time_window_key: str,
    time_window_label: str = '',
    label_points: bool = True,
) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    axes = axes.flatten()

    sns.set_style('white')

    for ax, (task_condition, rpe_cat) in zip(axes, conditions):
        panel_df = merged_df[
            (merged_df['task_condition'] == task_condition)
            & (merged_df['rpe_cat'] == rpe_cat)
        ].copy()
        panel_df = panel_df[np.isfinite(panel_df[gc_col]) & np.isfinite(panel_df[target_col])]

        ax.set_title(f'{task_condition} / {rpe_cat}')
        if len(panel_df) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            continue

        x = panel_df[gc_col].to_numpy(dtype=float)
        y = panel_df[target_col].to_numpy(dtype=float)

        sns.scatterplot(x=x, y=y, ax=ax, s=52, color='#4c78a8', edgecolor='black', linewidth=0.7)
        _fit_and_plot_line(ax, x, y)

        if label_points:
            for row in panel_df.itertuples(index=False):
                ax.text(getattr(row, gc_col), getattr(row, target_col), f' {row.Subject}', fontsize=8)

        res_match = results_df[
            (results_df['target_roi'] == target_label)
            & (results_df['time_window'] == time_window_key)
            & (results_df['task_condition'] == task_condition)
            & (results_df['rpe_cat'] == rpe_cat)
        ]
        if len(res_match) > 0:
            rr = res_match.iloc[0]
            ann = f"{rr['primary_corr_type']} r={rr['primary_r']:.3f}, p={rr['primary_p']:.3g}, n={int(rr['n'])}"
            ax.text(0.02, 0.98, ann, transform=ax.transAxes, ha='left', va='top', fontsize=10)

        ax.set_xlabel(gc_col)
        ax.set_ylabel(target_col)

        ax.grid(False)
        sns.despine(ax=ax, top=True, right=True)

    tw = time_window_label or time_window_key
    fig.suptitle(f'{target_label} vs {gc_col} by condition ({tw})', fontsize=14)
    out = Path(output_pdf)
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=300)
    plt.close(fig)


def average_gc_in_time_window(
    gc_df: pd.DataFrame,
    tmin: float,
    tmax: float,
    gc_roi_source: str,
    gc_roi_target: str,
    gc_metric: str,
) -> pd.DataFrame:
    m = (
        (gc_df['time'] >= tmin)
        & (gc_df['time'] <= tmax)
        & (gc_df['roi_source'] == gc_roi_source)
        & (gc_df['roi_target'] == gc_roi_target)
    )
    sub = gc_df.loc[m].copy()
    if len(sub) == 0:
        raise ValueError(
            f'No GC rows for {gc_roi_source}->{gc_roi_target} in time [{tmin}, {tmax}].'
        )
    out = (
        sub.groupby(['participant', 'rpe_cat'], as_index=False, dropna=False)
        .agg(**{gc_metric: (gc_metric, 'mean')})
    )
    out = out.rename(columns={'participant': 'Subject'})
    out['task_condition'] = 'cue'
    return out


def build_sliding_spearman_table(
    gc_long: pd.DataFrame,
    target_tbl: pd.DataFrame,
    target_col: str,
    gc_roi_source: str,
    gc_roi_target: str,
    gc_metric: str,
    rpe_cats: Tuple[str, ...],
    window_len_s: float,
    step_s: float,
    min_n: int,
) -> pd.DataFrame:
    g = gc_long.loc[
        (gc_long['roi_source'] == gc_roi_source)
        & (gc_long['roi_target'] == gc_roi_target)
    ].copy()

    times = np.sort(g['time'].unique())
    t_lo = float(np.min(times))
    t_hi = float(np.max(times))
    half = window_len_s / 2.0
    if t_hi - t_lo < window_len_s:
        raise ValueError(
            f'Epoch [{t_lo}, {t_hi}] s is shorter than sliding_window_len_s={window_len_s}.'
        )
    start_c = t_lo + half
    end_c = t_hi - half
    centers = np.arange(start_c, end_c + step_s * 0.5, step_s, dtype=float)
    centers = centers[centers <= end_c + 1e-9]

    target_df = target_tbl[['Subject', target_col]].copy()
    rows = []

    for center in centers:
        tmin = float(center - half)
        tmax = float(center + half)
        m = (g['time'] >= tmin) & (g['time'] <= tmax)
        sub_w = g.loc[m].copy()
        if len(sub_w) == 0:
            continue

        for cat in rpe_cats:
            sc = sub_w[sub_w['rpe_cat'] == cat].copy()
            if len(sc) == 0:
                rows.append({
                    'time': float(center),
                    'window_tmin': tmin,
                    'window_tmax': tmax,
                    'rpe_cat': cat,
                    'spearman_rho': np.nan,
                    'spearman_p': np.nan,
                    'se_rho': np.nan,
                    'mean_gc': np.nan,
                    'sem_mean_gc': np.nan,
                    'n_subjects': 0,
                })
                continue

            grp = (
                sc.groupby('participant', as_index=False, dropna=False)
                .agg(**{gc_metric: (gc_metric, 'mean')})
            )
            grp = grp.rename(columns={'participant': 'Subject'})
            merged = grp.merge(target_df, on='Subject', how='inner')
            x = merged[gc_metric].to_numpy(dtype=float)
            y = merged[target_col].to_numpy(dtype=float)
            mask = np.isfinite(x) & np.isfinite(y)
            x = x[mask]
            y = y[mask]
            n = int(len(x))
            mean_gc = float(np.nanmean(x)) if n > 0 else np.nan
            if n >= 2:
                sem_mean_gc = float(np.std(x, ddof=1) / np.sqrt(n))
            else:
                sem_mean_gc = np.nan
            if n < min_n:
                rows.append({
                    'time': float(center),
                    'window_tmin': tmin,
                    'window_tmax': tmax,
                    'rpe_cat': cat,
                    'spearman_rho': np.nan,
                    'spearman_p': np.nan,
                    'se_rho': np.nan,
                    'mean_gc': mean_gc,
                    'sem_mean_gc': sem_mean_gc,
                    'n_subjects': n,
                })
                continue
            rho, pval = spearmanr(x, y)
            rows.append({
                'time': float(center),
                'window_tmin': tmin,
                'window_tmax': tmax,
                'rpe_cat': cat,
                'spearman_rho': float(rho),
                'spearman_p': float(pval),
                'se_rho': np.nan,
                'mean_gc': mean_gc,
                'sem_mean_gc': sem_mean_gc,
                'n_subjects': n,
            })
    return pd.DataFrame(rows)


def load_roi_electrode_counts(subjects: List[str], base_dir: str) -> pd.DataFrame:
    rows = []
    missing_elec_tables = []
    for subj_id in subjects:
        elec_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df'
        if not os.path.isfile(elec_path):
            missing_elec_tables.append(elec_path)
            continue
        elec_df = pd.read_csv(elec_path)
        region_col = 'salman_region' if 'salman_region' in elec_df.columns else 'region' if 'region' in elec_df.columns else None
        if region_col is None:
            raise ValueError(f'Could not identify ROI column for {subj_id}.')
        rows.append({
            'Subject': subj_id,
            'ACC_NumElectrodes': int(elec_df.loc[elec_df[region_col].astype(str) == 'ACC', 'label'].astype(str).nunique()),
            'HPC_NumElectrodes': int(elec_df.loc[elec_df[region_col].astype(str) == 'HPC', 'label'].astype(str).nunique()),
        })
    if missing_elec_tables:
        print(f'Missing electrode tables for {len(missing_elec_tables)} subject(s).')
    return pd.DataFrame(rows).sort_values('Subject').reset_index(drop=True)


def plot_subject_covariate_grid(
    merged_df: pd.DataFrame,
    results_df: pd.DataFrame,
    target_col: str,
    predictor_order: List[str],
    predictor_labels: Dict[str, str],
    output_pdf: Path,
    label_points: bool = True,
) -> None:
    n_panels = len(predictor_order)
    ncols = 3
    nrows = int(np.ceil(n_panels / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 5 * nrows), constrained_layout=True)
    axes = np.atleast_1d(axes).flatten()
    sns.set_style('white')

    for ax, predictor in zip(axes, predictor_order):
        panel_df = merged_df[np.isfinite(merged_df[predictor]) & np.isfinite(merged_df[target_col])].copy()
        ax.set_title(predictor_labels[predictor])
        if len(panel_df) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            continue

        x = panel_df[predictor].to_numpy(dtype=float)
        y = panel_df[target_col].to_numpy(dtype=float)
        sns.scatterplot(x=x, y=y, ax=ax, s=52, color='#4c78a8', edgecolor='black', linewidth=0.7)
        _fit_and_plot_line(ax, x, y)

        if label_points:
            for row in panel_df.itertuples(index=False):
                ax.text(getattr(row, predictor), getattr(row, target_col), f' {row.Subject}', fontsize=8)

        rr = results_df[results_df['predictor'] == predictor].iloc[0]
        ann = (
            f"{rr['primary_corr_type']} r={rr['primary_r']:.3f}, "
            f"p={rr['primary_p']:.3g}, n={int(rr['n'])}"
        )
        ax.text(0.02, 0.98, ann, transform=ax.transAxes, ha='left', va='top', fontsize=10)
        ax.set_xlabel(predictor_labels[predictor])
        ax.set_ylabel(target_col)
        ax.grid(False)
        sns.despine(ax=ax, top=True, right=True)

    for ax in axes[n_panels:]:
        ax.axis('off')

    fig.suptitle(f'{rsa_target_label} vs subject-level covariates', fontsize=14)
    out = Path(output_pdf)
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=300)
    plt.close(fig)


In [ ]:
# ---- Load ACC→HPC RSA results and reconstruct the significant diff cluster ----
if not os.path.isfile(rsa_group_results_pkl):
    raise FileNotFoundError(f'RSA group results PKL not found: {rsa_group_results_pkl}')

with open(rsa_group_results_pkl, 'rb') as f:
    rsa_group_results = pickle.load(f)

required_rsa_keys = [
    'observed_diffs_stack',
    'feedback_times',
    'cue_times',
    'subject_results',
    'successful_subjects',
    'n_subjects',
]
missing_rsa_keys = [k for k in required_rsa_keys if k not in rsa_group_results]
if missing_rsa_keys:
    raise ValueError(f'RSA group results missing required keys: {missing_rsa_keys}')

X = np.asarray(rsa_group_results['observed_diffs_stack'], dtype=float)
feedback_times = np.asarray(rsa_group_results['feedback_times'], dtype=float)
cue_times = np.asarray(rsa_group_results['cue_times'], dtype=float)

if X.ndim != 3:
    raise ValueError(f'Expected observed_diffs_stack to be 3D; got shape {X.shape}.')
if X.shape[1] != len(feedback_times) or X.shape[2] != len(cue_times):
    raise ValueError(
        'Mismatch between observed_diffs_stack and feedback/cue time dimensions: '
        f'X={X.shape}, feedback={len(feedback_times)}, cue={len(cue_times)}'
    )

n_fb = X.shape[1]
n_cue = X.shape[2]

fb_adjacency = mne.stats.combine_adjacency(n_fb)
cue_adjacency = mne.stats.combine_adjacency(n_cue)
adjacency_2d = csr_matrix(kron(eye(n_fb), cue_adjacency) + kron(fb_adjacency, eye(n_cue)))
X_flat = X.reshape(X.shape[0], -1)

t_obs, clusters, cluster_pv, H0 = permutation_cluster_1samp_test(
    X_flat,
    n_permutations=cluster_n_permutations,
    threshold=None,
    adjacency=adjacency_2d,
    tail=cluster_tail,
    n_jobs=8,
    seed=cluster_seed,
    verbose=False,
)

sig_cluster_indices = [i for i, pval in enumerate(cluster_pv) if pval < cluster_alpha]
if len(sig_cluster_indices) != 1:
    raise ValueError(
        f'Expected exactly 1 significant ACC→HPC cluster at alpha={cluster_alpha}; '
        f'found {len(sig_cluster_indices)}.'
    )

sig_cluster_index = sig_cluster_indices[0]
sig_cluster = clusters[sig_cluster_index]
sig_mask_flat = np.zeros(n_fb * n_cue, dtype=bool)
sig_mask_flat[sig_cluster[0]] = True
sig_cluster_mask = sig_mask_flat.reshape(n_fb, n_cue)
sig_cluster_size = int(sig_cluster_mask.sum())

fb_indices = np.where(sig_cluster_mask.any(axis=1))[0]
cue_indices = np.where(sig_cluster_mask.any(axis=0))[0]
if len(fb_indices) == 0 or len(cue_indices) == 0:
    raise ValueError('Significant cluster mask is empty after reshaping.')

cluster_metadata = {
    'feedback_roi': rsa_feedback_roi,
    'cue_roi': rsa_cue_roi,
    'cluster_index_zero_based': int(sig_cluster_index),
    'cluster_p_value': float(cluster_pv[sig_cluster_index]),
    'cluster_size': sig_cluster_size,
    'n_cluster_subjects': int(X.shape[0]),
    'n_feedback_timepoints': int(n_fb),
    'n_cue_timepoints': int(n_cue),
    'feedback_tmin': float(feedback_times[fb_indices[0]]),
    'feedback_tmax': float(feedback_times[fb_indices[-1]]),
    'cue_tmin': float(cue_times[cue_indices[0]]),
    'cue_tmax': float(cue_times[cue_indices[-1]]),
    'n_feedback_rows_in_mask': int(len(fb_indices)),
    'n_cue_cols_in_mask': int(len(cue_indices)),
    'mask_bbox_size': int(len(fb_indices) * len(cue_indices)),
    'mask_fraction_in_bbox': float(sig_cluster_size / (len(fb_indices) * len(cue_indices))),
    'cluster_alpha': float(cluster_alpha),
    'cluster_n_permutations': int(cluster_n_permutations),
    'cluster_tail': int(cluster_tail),
    'cluster_seed': int(cluster_seed),
}
cluster_metadata_df = pd.DataFrame([cluster_metadata])

print('ACC→HPC cluster test complete.')
print('Significant clusters found:', len(sig_cluster_indices))
display(cluster_metadata_df)


In [ ]:
# ---- Build subject-level ACC→HPC hit RSA within the significant cluster ----
rsa_rows = []
for subject_result in rsa_group_results['subject_results']:
    observed = subject_result['observed']
    subject = observed['subject']

    category_heatmaps = observed.get('category_heatmaps', {})
    if hit_memory_label not in category_heatmaps:
        raise ValueError(f'Subject {subject} missing category_heatmaps[{hit_memory_label!r}].')

    hit_heatmap = np.asarray(category_heatmaps[hit_memory_label], dtype=float)
    if hit_heatmap.shape != sig_cluster_mask.shape:
        raise ValueError(
            f'Subject {subject} heatmap shape {hit_heatmap.shape} does not match '
            f'cluster mask shape {sig_cluster_mask.shape}.'
        )

    diff_heatmap = np.asarray(observed['diff_heatmap'], dtype=float)
    n_trials_per_category = observed.get('n_trials_per_category', {})

    rsa_rows.append({
        'Subject': subject,
        rsa_metric_col: float(np.nanmean(hit_heatmap[sig_cluster_mask])),
        'ACC_HPC_DiffRSA_cluster_mean': float(np.nanmean(diff_heatmap[sig_cluster_mask])),
        'ACC_HPC_RememberedTrialCount': int(n_trials_per_category.get('remembered', 0)),
        'ACC_HPC_ForgottenTrialCount': int(n_trials_per_category.get('forgotten', 0)),
        'ACC_HPC_TotalTrialCount': int(observed.get('n_trials', 0)),
    })

rsa_subject_tbl = (
    pd.DataFrame(rsa_rows)
    .sort_values('Subject')
    .reset_index(drop=True)
)
assert_required_columns(
    rsa_subject_tbl,
    ['Subject', rsa_metric_col, 'ACC_HPC_RememberedTrialCount', 'ACC_HPC_ForgottenTrialCount'],
    'ACC-HPC hit RSA subject table',
)

print('Subjects in ACC→HPC hit RSA table:', rsa_subject_tbl['Subject'].nunique())
display(rsa_subject_tbl.head())


In [ ]:
# ---- Load HPC HFA reinstatement and merge with the ACC-HPC hit RSA cohort ----
if not os.path.isfile(hpc_hfa_reinstatement_csv):
    raise FileNotFoundError(f'HPC HFA reinstatement CSV not found: {hpc_hfa_reinstatement_csv}')

hpc_hfa_tbl = pd.read_csv(hpc_hfa_reinstatement_csv)
assert_required_columns(
    hpc_hfa_tbl,
    ['Subject', 'MeanAccuracy', 'NumElectrodes'],
    'HPC HFA reinstatement table',
)

hpc_hfa_metric_col = 'HPC_hfa_MeanAccuracy'
hpc_hfa_tbl = hpc_hfa_tbl.rename(
    columns={
        'MeanAccuracy': hpc_hfa_metric_col,
        'NumElectrodes': 'HPC_hfa_NumElectrodes',
    }
)[['Subject', hpc_hfa_metric_col, 'HPC_hfa_NumElectrodes']]

hpc_hfa_rsa_tbl = deterministic_inner_join(
    rsa_subject_tbl,
    hpc_hfa_tbl,
    keys=['Subject'],
)
if len(hpc_hfa_rsa_tbl) == 0:
    raise ValueError('No overlapping subjects after merging RSA and HPC HFA reinstatement.')

hpc_hfa_rsa_stats = run_correlation_analysis(
    x=hpc_hfa_rsa_tbl[hpc_hfa_metric_col].to_numpy(dtype=float),
    y=hpc_hfa_rsa_tbl[rsa_metric_col].to_numpy(dtype=float),
    corr_type=corr_type,
    run_secondary_pearson=run_secondary_pearson,
)
hpc_hfa_rsa_results_df = pd.DataFrame([{
    'analysis_label': 'HPC HFA MeanAccuracy vs ACC-HPC Hit RSA',
    'predictor': hpc_hfa_metric_col,
    'target': rsa_metric_col,
    'n': hpc_hfa_rsa_stats['n'],
    'spearman_rho': hpc_hfa_rsa_stats['spearman_rho'],
    'spearman_p': hpc_hfa_rsa_stats['spearman_p'],
    'pearson_r': hpc_hfa_rsa_stats['pearson_r'],
    'pearson_p': hpc_hfa_rsa_stats['pearson_p'],
    'primary_corr_type': hpc_hfa_rsa_stats['primary_corr_type'],
    'primary_r': hpc_hfa_rsa_stats['primary_r'],
    'primary_p': hpc_hfa_rsa_stats['primary_p'],
}])

print('Subjects in RSA table:', rsa_subject_tbl['Subject'].nunique())
print('Subjects in HPC HFA reinstatement table:', hpc_hfa_tbl['Subject'].nunique())
print('Subjects in RSA + HPC HFA merged table:', hpc_hfa_rsa_tbl['Subject'].nunique())
display(hpc_hfa_rsa_tbl.head())
display(hpc_hfa_rsa_results_df)


In [ ]:
# ---- Correlate HPC theta MeanAccuracy with HPC HFA MeanAccuracy ----
hpc_theta_hfa_tbl = deterministic_inner_join(
    hpc_theta_rsa_tbl[['Subject', hpc_theta_metric_col]],
    hpc_hfa_rsa_tbl[['Subject', hpc_hfa_metric_col]],
    keys=['Subject'],
)
if len(hpc_theta_hfa_tbl) == 0:
    raise ValueError('No overlapping subjects after merging HPC theta and HPC HFA tables.')

hpc_theta_hfa_stats = run_correlation_analysis(
    x=hpc_theta_hfa_tbl[hpc_theta_metric_col].to_numpy(dtype=float),
    y=hpc_theta_hfa_tbl[hpc_hfa_metric_col].to_numpy(dtype=float),
    corr_type=corr_type,
    run_secondary_pearson=run_secondary_pearson,
)
hpc_theta_hfa_results_df = pd.DataFrame([{
    'analysis_label': 'HPC theta MeanAccuracy vs HPC HFA MeanAccuracy',
    'predictor': hpc_theta_metric_col,
    'target': hpc_hfa_metric_col,
    'n': hpc_theta_hfa_stats['n'],
    'spearman_rho': hpc_theta_hfa_stats['spearman_rho'],
    'spearman_p': hpc_theta_hfa_stats['spearman_p'],
    'pearson_r': hpc_theta_hfa_stats['pearson_r'],
    'pearson_p': hpc_theta_hfa_stats['pearson_p'],
    'primary_corr_type': hpc_theta_hfa_stats['primary_corr_type'],
    'primary_r': hpc_theta_hfa_stats['primary_r'],
    'primary_p': hpc_theta_hfa_stats['primary_p'],
}])

print('Subjects in HPC theta + HPC HFA merged table:', hpc_theta_hfa_tbl['Subject'].nunique())
display(hpc_theta_hfa_tbl.head())
print('HPC theta MeanAccuracy vs HPC HFA MeanAccuracy:')
display(hpc_theta_hfa_results_df)


In [ ]:
# ---- Load ACC reactivation and merge with the ACC-HPC hit RSA cohort ----
if not os.path.isfile(acc_reinstatement_csv):
    raise FileNotFoundError(f'ACC reinstatement CSV not found: {acc_reinstatement_csv}')

acc_reactivation_tbl = pd.read_csv(acc_reinstatement_csv)
assert_required_columns(
    acc_reactivation_tbl,
    ['Subject', acc_reactivation_metric, 'NumElectrodes'],
    'ACC reinstatement table',
)

acc_reactivation_tbl = acc_reactivation_tbl.rename(
    columns={
        acc_reactivation_metric: acc_reactivation_col,
        'NumElectrodes': 'ACC_NumElectrodes',
    }
)[['Subject', acc_reactivation_col, 'ACC_NumElectrodes']]

acc_reactivation_rsa_tbl = deterministic_inner_join(
    acc_reactivation_tbl,
    rsa_subject_tbl,
    keys=['Subject'],
)

expected_rsa_subjects = sorted(rsa_subject_tbl['Subject'].tolist())
merged_rsa_subjects = sorted(acc_reactivation_rsa_tbl['Subject'].tolist())
if merged_rsa_subjects != expected_rsa_subjects:
    raise ValueError(
        'ACC reactivation merge did not preserve the RSA cohort exactly. '
        f'Expected {expected_rsa_subjects}, got {merged_rsa_subjects}.'
    )

print('Subjects in ACC reactivation table:', acc_reactivation_tbl['Subject'].nunique())
print('Subjects in ACC reactivation + RSA merged table:', acc_reactivation_rsa_tbl['Subject'].nunique())
display(acc_reactivation_rsa_tbl.head())


In [ ]:
# ---- Load HPC theta reinstatement and merge with the ACC-HPC hit RSA cohort ----
if not os.path.isfile(hpc_theta_reinstatement_csv):
    raise FileNotFoundError(f'HPC theta reinstatement CSV not found: {hpc_theta_reinstatement_csv}')

hpc_theta_tbl = pd.read_csv(hpc_theta_reinstatement_csv)
assert_required_columns(
    hpc_theta_tbl,
    ['Subject', 'MeanAccuracy', 'NumElectrodes'],
    'HPC theta reinstatement table',
)

hpc_theta_metric_col = 'HPC_theta_MeanAccuracy'
hpc_theta_tbl = hpc_theta_tbl.rename(
    columns={
        'MeanAccuracy': hpc_theta_metric_col,
        'NumElectrodes': 'HPC_theta_NumElectrodes',
    }
)[['Subject', hpc_theta_metric_col, 'HPC_theta_NumElectrodes']]

hpc_theta_rsa_tbl = deterministic_inner_join(
    rsa_subject_tbl,
    hpc_theta_tbl,
    keys=['Subject'],
)
if len(hpc_theta_rsa_tbl) == 0:
    raise ValueError('No overlapping subjects after merging RSA and HPC theta reinstatement.')

hpc_theta_rsa_stats = run_correlation_analysis(
    x=hpc_theta_rsa_tbl[hpc_theta_metric_col].to_numpy(dtype=float),
    y=hpc_theta_rsa_tbl[rsa_metric_col].to_numpy(dtype=float),
    corr_type=corr_type,
    run_secondary_pearson=run_secondary_pearson,
)
hpc_theta_rsa_results_df = pd.DataFrame([{
    'analysis_label': 'HPC theta MeanAccuracy vs ACC-HPC Hit RSA',
    'predictor': hpc_theta_metric_col,
    'target': rsa_metric_col,
    'n': hpc_theta_rsa_stats['n'],
    'spearman_rho': hpc_theta_rsa_stats['spearman_rho'],
    'spearman_p': hpc_theta_rsa_stats['spearman_p'],
    'pearson_r': hpc_theta_rsa_stats['pearson_r'],
    'pearson_p': hpc_theta_rsa_stats['pearson_p'],
    'primary_corr_type': hpc_theta_rsa_stats['primary_corr_type'],
    'primary_r': hpc_theta_rsa_stats['primary_r'],
    'primary_p': hpc_theta_rsa_stats['primary_p'],
}])

print('Subjects in RSA table:', rsa_subject_tbl['Subject'].nunique())
print('Subjects in HPC theta reinstatement table:', hpc_theta_tbl['Subject'].nunique())
print('Subjects in RSA + HPC theta merged table:', hpc_theta_rsa_tbl['Subject'].nunique())
display(hpc_theta_rsa_tbl.head())
display(hpc_theta_rsa_results_df)


In [ ]:
# ---- Run subject-level ACC-HPC hit RSA vs subject covariates ----
if not os.path.isfile(learn_df_csv):
    raise FileNotFoundError(f'Behavior CSV not found: {learn_df_csv}')
if not os.path.isfile(combined_df_csv):
    raise FileNotFoundError(f'Combined CSV not found: {combined_df_csv}')

learn_df = pd.read_csv(learn_df_csv)
combined_df = pd.read_csv(combined_df_csv)
assert_required_columns(learn_df, ['participant', 'reward', 'alpha', 'beta'], 'learn_df')
assert_required_columns(combined_df, ['participant', 'condition', 'dprime'], 'combined_df')

behavior_consistency_tbl = (
    learn_df.groupby('participant', as_index=False)
    .agg(
        alpha_nunique=('alpha', lambda s: s.dropna().nunique()),
        beta_nunique=('beta', lambda s: s.dropna().nunique()),
    )
    .sort_values('participant')
    .reset_index(drop=True)
)
alpha_conflicts = behavior_consistency_tbl[behavior_consistency_tbl['alpha_nunique'] > 1]
beta_conflicts = behavior_consistency_tbl[behavior_consistency_tbl['beta_nunique'] > 1]
if len(alpha_conflicts) > 0 or len(beta_conflicts) > 0:
    raise ValueError(
        'Found conflicting participant-level alpha/beta values in learn_df. '
        f'alpha_conflicts={alpha_conflicts.to_dict(orient="records")}, '
        f'beta_conflicts={beta_conflicts.to_dict(orient="records")}'
    )
print('Behavior parameter consistency audit:')
display(behavior_consistency_tbl.head())

behavior_tbl = (
    learn_df.groupby('participant', as_index=False)
    .agg(
        total_reward=('reward', lambda s: s.sum(min_count=1)),
        alpha=('alpha', lambda s: s.dropna().iloc[0] if len(s.dropna()) > 0 else np.nan),
        beta=('beta', lambda s: s.dropna().iloc[0] if len(s.dropna()) > 0 else np.nan),
    )
    .rename(columns={'participant': 'Subject'})
    .sort_values('Subject')
    .reset_index(drop=True)
)

combined_df_day1 = combined_df[combined_df['condition'] == combined_df_condition].copy()
dprime_consistency_tbl = (
    combined_df_day1.groupby('participant', as_index=False)
    .agg(dprime_nunique=('dprime', lambda s: s.dropna().nunique()))
    .sort_values('participant')
    .reset_index(drop=True)
)
dprime_conflicts = dprime_consistency_tbl[dprime_consistency_tbl['dprime_nunique'] > 1]
if len(dprime_conflicts) > 0:
    raise ValueError(
        'Found conflicting participant-level dprime values in combined_df after condition filtering. '
        f'dprime_conflicts={dprime_conflicts.to_dict(orient="records")}'
    )
print('\nd-prime consistency audit (condition-filtered):')
display(dprime_consistency_tbl.head())

dprime_tbl = (
    combined_df_day1.groupby('participant', as_index=False)
    .agg(dprime=('dprime', lambda s: s.dropna().iloc[0] if len(s.dropna()) > 0 else np.nan))
    .rename(columns={'participant': 'Subject'})
    .sort_values('Subject')
    .reset_index(drop=True)
)

electrode_count_tbl = load_roi_electrode_counts(rsa_subject_tbl['Subject'].tolist(), base_dir)
print('\nElectrode count coverage:')
display(electrode_count_tbl.head())

rsa_subject_covariate_tbl = deterministic_inner_join(rsa_subject_tbl, behavior_tbl, keys=['Subject'])
rsa_subject_covariate_tbl = deterministic_inner_join(rsa_subject_covariate_tbl, dprime_tbl, keys=['Subject'])
rsa_subject_covariate_tbl = deterministic_inner_join(rsa_subject_covariate_tbl, electrode_count_tbl, keys=['Subject'])

expected_rsa_subjects = sorted(rsa_subject_tbl['Subject'].tolist())
merged_covariate_subjects = sorted(rsa_subject_covariate_tbl['Subject'].tolist())
if merged_covariate_subjects != expected_rsa_subjects:
    raise ValueError(
        'Subject covariate merge did not preserve the RSA cohort exactly. '
        f'Expected {expected_rsa_subjects}, got {merged_covariate_subjects}.'
    )

subject_covariate_missing_audit = rsa_subject_covariate_tbl[['Subject', rsa_metric_col] + subject_covariate_predictor_order].isna().sum()
subject_covariate_counts = []
for predictor in subject_covariate_predictor_order:
    n_valid = int(np.isfinite(rsa_subject_covariate_tbl[predictor].to_numpy(dtype=float)).sum())
    subject_covariate_counts.append({'predictor': predictor, 'n_valid_subjects': n_valid})
    if n_valid < 3:
        raise ValueError(f'Predictor {predictor} has fewer than 3 finite values.')
subject_covariate_counts_df = pd.DataFrame(subject_covariate_counts)

subject_covariate_result_rows = []
for predictor in subject_covariate_predictor_order:
    stats = run_correlation_analysis(
        x=rsa_subject_covariate_tbl[predictor].to_numpy(dtype=float),
        y=rsa_subject_covariate_tbl[rsa_metric_col].to_numpy(dtype=float),
        corr_type=corr_type,
        run_secondary_pearson=run_secondary_pearson,
    )
    subject_covariate_result_rows.append({
        'predictor': predictor,
        'predictor_label': subject_covariate_predictor_labels[predictor],
        'target': rsa_metric_col,
        'n': stats['n'],
        'spearman_rho': stats['spearman_rho'],
        'spearman_p': stats['spearman_p'],
        'pearson_r': stats['pearson_r'],
        'pearson_p': stats['pearson_p'],
        'primary_corr_type': stats['primary_corr_type'],
        'primary_r': stats['primary_r'],
        'primary_p': stats['primary_p'],
        'metric_name': rsa_metric_name,
    })

subject_covariate_results_df = pd.DataFrame(subject_covariate_result_rows)
subject_covariate_results_df['predictor'] = pd.Categorical(
    subject_covariate_results_df['predictor'],
    categories=subject_covariate_predictor_order,
    ordered=True,
)
subject_covariate_results_df = subject_covariate_results_df.sort_values('predictor').reset_index(drop=True)
if len(subject_covariate_results_df) != 6:
    raise ValueError(f'Expected 6 subject-covariate correlation rows; found {len(subject_covariate_results_df)}.')

print('\nSubjects in RSA + subject-covariate table:', rsa_subject_covariate_tbl['Subject'].nunique())
print('Subject-covariate counts:')
display(subject_covariate_counts_df)
print('\nSubject-covariate missing-value audit:')
display(subject_covariate_missing_audit.to_frame(name='n_missing').T)
print('\nSubject-covariate merged table preview:')
display(rsa_subject_covariate_tbl.head())
print('\nACC-HPC hit RSA vs subject covariates:')
display(subject_covariate_results_df)


In [ ]:
# ---- Run subject-level ACC reactivation vs ACC-HPC hit RSA correlation ----
acc_reactivation_rsa_stats = run_correlation_analysis(
    x=acc_reactivation_rsa_tbl[acc_reactivation_col].to_numpy(dtype=float),
    y=acc_reactivation_rsa_tbl[rsa_metric_col].to_numpy(dtype=float),
    corr_type=corr_type,
    run_secondary_pearson=run_secondary_pearson,
)

acc_reactivation_rsa_results_df = pd.DataFrame([{
    'analysis_label': acc_reactivation_analysis_label,
    'predictor': acc_reactivation_col,
    'target': rsa_metric_col,
    'n': acc_reactivation_rsa_stats['n'],
    'spearman_rho': acc_reactivation_rsa_stats['spearman_rho'],
    'spearman_p': acc_reactivation_rsa_stats['spearman_p'],
    'pearson_r': acc_reactivation_rsa_stats['pearson_r'],
    'pearson_p': acc_reactivation_rsa_stats['pearson_p'],
    'primary_corr_type': acc_reactivation_rsa_stats['primary_corr_type'],
    'primary_r': acc_reactivation_rsa_stats['primary_r'],
    'primary_p': acc_reactivation_rsa_stats['primary_p'],
}])

print('ACC reactivation vs ACC-HPC hit RSA:')
display(acc_reactivation_rsa_results_df)


In [ ]:
# ---- Save merged tables, correlation tables, subject metrics, and cluster metadata ----
rsa_subject_tbl.to_csv(paths['subject_metric_csv'], index=False)
cluster_metadata_df.to_csv(paths['cluster_metadata_csv'], index=False)
with open(paths['cluster_metadata_json'], 'w', encoding='utf-8') as f:
    json.dump(cluster_metadata, f, indent=2)

acc_reactivation_rsa_tbl.to_csv(paths['acc_reactivation_merge_csv'], index=False)
acc_reactivation_rsa_results_df.to_csv(paths['acc_reactivation_results_csv'], index=False)

rsa_subject_covariate_tbl.to_csv(paths['subject_covariate_merge_csv'], index=False)
subject_covariate_results_df.to_csv(paths['subject_covariate_results_csv'], index=False)

print('Saved subject metrics & correlation tables to:', analysis_output_root)


In [ ]:
# ---- Save ACC reactivation vs ACC-HPC hit RSA scatter ----
fig, ax = plt.subplots(1, 1, figsize=(6, 5), constrained_layout=True)

plot_df = acc_reactivation_rsa_tbl[
    np.isfinite(acc_reactivation_rsa_tbl[acc_reactivation_col])
    & np.isfinite(acc_reactivation_rsa_tbl[rsa_metric_col])
].copy()

x = plot_df[acc_reactivation_col].to_numpy(dtype=float)
y = plot_df[rsa_metric_col].to_numpy(dtype=float)

sns.set_style('white')
sns.scatterplot(x=x, y=y, ax=ax, s=52, color='#4c78a8', edgecolor='black', linewidth=0.7)
_fit_and_plot_line(ax, x, y)

if label_points:
    for row in plot_df.itertuples(index=False):
        ax.text(getattr(row, acc_reactivation_col), getattr(row, rsa_metric_col), f' {row.Subject}', fontsize=8)

rr = acc_reactivation_rsa_results_df.iloc[0]
ann = f"{rr['primary_corr_type']} r={rr['primary_r']:.3f}, p={rr['primary_p']:.3g}, n={int(rr['n'])}"
ax.text(0.02, 0.98, ann, transform=ax.transAxes, ha='left', va='top', fontsize=10)

ax.set_xlabel(acc_reactivation_col)
ax.set_ylabel(rsa_metric_col)
ax.set_title(acc_reactivation_analysis_label)
ax.grid(False)
sns.despine(ax=ax, top=True, right=True)

Path(paths['acc_reactivation_scatter_pdf']).parent.mkdir(parents=True, exist_ok=True)
fig.savefig(paths['acc_reactivation_scatter_pdf'], dpi=300)
print('Saved ACC reactivation scatter plot:', paths['acc_reactivation_scatter_pdf'])


In [ ]:
# ---- Save HPC theta MeanAccuracy vs ACC-HPC hit RSA scatter ----
fig, ax = plt.subplots(1, 1, figsize=(6, 5), constrained_layout=True)

plot_df = hpc_theta_rsa_tbl[
    np.isfinite(hpc_theta_rsa_tbl[hpc_theta_metric_col])
    & np.isfinite(hpc_theta_rsa_tbl[rsa_metric_col])
].copy()

x = plot_df[hpc_theta_metric_col].to_numpy(dtype=float)
y = plot_df[rsa_metric_col].to_numpy(dtype=float)

sns.set_style('white')
sns.scatterplot(x=x, y=y, ax=ax, s=52, color='#4c78a8', edgecolor='black', linewidth=0.7)
_fit_and_plot_line(ax, x, y)

if label_points:
    for row in plot_df.itertuples(index=False):
        ax.text(getattr(row, hpc_theta_metric_col), getattr(row, rsa_metric_col), f' {row.Subject}', fontsize=8)

rr = hpc_theta_rsa_results_df.iloc[0]
ann = f"{rr['primary_corr_type']} r={rr['primary_r']:.3f}, p={rr['primary_p']:.3g}, n={int(rr['n'])}"
ax.text(0.02, 0.98, ann, transform=ax.transAxes, ha='left', va='top', fontsize=10)

ax.set_xlabel(hpc_theta_metric_col)
ax.set_ylabel(rsa_metric_col)
ax.set_title('HPC theta MeanAccuracy vs ACC-HPC Hit RSA')
ax.grid(False)
sns.despine(ax=ax, top=True, right=True)

Path(paths['hpc_theta_scatter_pdf']).parent.mkdir(parents=True, exist_ok=True)
fig.savefig(paths['hpc_theta_scatter_pdf'], dpi=300)
print('Saved HPC theta MeanAccuracy scatter plot:', paths['hpc_theta_scatter_pdf'])


In [ ]:
# ---- Save HPC HFA MeanAccuracy vs ACC-HPC hit RSA scatter ----
fig, ax = plt.subplots(1, 1, figsize=(6, 5), constrained_layout=True)

plot_df = hpc_hfa_rsa_tbl[
    np.isfinite(hpc_hfa_rsa_tbl[hpc_hfa_metric_col])
    & np.isfinite(hpc_hfa_rsa_tbl[rsa_metric_col])
].copy()

x = plot_df[hpc_hfa_metric_col].to_numpy(dtype=float)
y = plot_df[rsa_metric_col].to_numpy(dtype=float)

sns.set_style('white')
sns.scatterplot(x=x, y=y, ax=ax, s=52, color='#4c78a8', edgecolor='black', linewidth=0.7)
_fit_and_plot_line(ax, x, y)

if label_points:
    for row in plot_df.itertuples(index=False):
        ax.text(getattr(row, hpc_hfa_metric_col), getattr(row, rsa_metric_col), f' {row.Subject}', fontsize=8)

rr = hpc_hfa_rsa_results_df.iloc[0]
ann = f"{rr['primary_corr_type']} r={rr['primary_r']:.3f}, p={rr['primary_p']:.3g}, n={int(rr['n'])}"
ax.text(0.02, 0.98, ann, transform=ax.transAxes, ha='left', va='top', fontsize=10)

ax.set_xlabel(hpc_hfa_metric_col)
ax.set_ylabel(rsa_metric_col)
ax.set_title('HPC HFA MeanAccuracy vs ACC-HPC Hit RSA')
ax.grid(False)
sns.despine(ax=ax, top=True, right=True)

Path(paths['hpc_hfa_scatter_pdf']).parent.mkdir(parents=True, exist_ok=True)
fig.savefig(paths['hpc_hfa_scatter_pdf'], dpi=300)
print('Saved HPC HFA MeanAccuracy scatter plot:', paths['hpc_hfa_scatter_pdf'])


In [ ]:
# ---- Save ACC-HPC hit RSA vs subject covariates scatter grid ----
plot_subject_covariate_grid(
    merged_df=rsa_subject_covariate_tbl,
    results_df=subject_covariate_results_df,
    target_col=rsa_metric_col,
    predictor_order=subject_covariate_predictor_order,
    predictor_labels=subject_covariate_predictor_labels,
    output_pdf=paths['subject_covariate_scatter_pdf'],
    label_points=label_points,
)

print('Saved ACC-HPC hit RSA subject covariate scatter plot:', paths['subject_covariate_scatter_pdf'])
